In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class WavetableRead(Scene):
    """Reading a wavetable: the gold pointer advances by a fractional phase
    increment per output sample and wraps modulo M. First beat reads the
    nearest entry at or below the pointer, so the output snaps to table
    values; second beat blends the two neighbors and the output lands close
    to the true sine."""

    def construct(self):
        M = 16
        DM = 2.4                      # phase increment, table indices/sample
        N = 16                        # output samples per beat
        table = np.sin(2 * np.pi * np.arange(M) / M)

        def read_nearest(p):
            return table[int(p) % M]

        def read_interp(p):
            i = int(p) % M
            a = p - int(p)
            return (1 - a) * table[i] + a * table[(i + 1) % M]

        # top panel: the table
        top = Axes(x_range=[0, M, 4], y_range=[-1.3, 1.3, 1],
                   x_length=10.6, y_length=2.4,
                   axis_config={"color": anim.IRON, "include_ticks": False,
                                "stroke_width": 1.5},
                   tips=False).move_to([-0.4, 1.85, 0])
        stems = VGroup(*[Line(top.c2p(m, 0), top.c2p(m, table[m]),
                              color=anim.IRON, stroke_width=2)
                         for m in range(M)])
        dots = VGroup(*[Dot(top.c2p(m, table[m]), radius=0.055,
                            color=anim.STEEL, stroke_color=anim.IRON,
                            stroke_width=1.2) for m in range(M)])
        table_label = MathTex(r"\texttt{table}[m]").scale(0.6)
        table_label.move_to(top.c2p(12.8, 1.0))
        zero_label = MathTex("m = 0").scale(0.5).next_to(top.c2p(0, -1.3), DOWN, buff=0.12)
        m_label = MathTex("M").scale(0.5).next_to(top.c2p(M, -1.3), DOWN, buff=0.12)
        dm_label = MathTex(r"\Delta m = 2.4", color=anim.GOLD).scale(0.66)
        dm_label.move_to([-5.4, -0.08, 0])

        # the wrap: an arc from the right edge back to the start
        wrap = CurvedArrow(top.c2p(M, 1.4), top.c2p(0.3, 1.4),
                           angle=0.3, color=anim.IRON, stroke_width=2,
                           tip_length=0.14)
        wrap_label = MathTex(r"p \bmod M").scale(0.5)
        wrap_label.move_to([-0.4, 3.62, 0])

        # the pointer, driven by accumulated phase p
        p = ValueTracker(0.0)
        pointer = always_redraw(lambda: Line(
            top.c2p(p.get_value() % M, -1.25),
            top.c2p(p.get_value() % M, 1.25),
            color=anim.GOLD, stroke_width=3))
        p_num = DecimalNumber(0.0, num_decimal_places=1).scale(0.5)
        p_eq = MathTex("p = ").scale(0.55)
        p_read = VGroup(p_eq, p_num).arrange(RIGHT, buff=0.1)
        p_read.move_to([-3.3, -0.08, 0])
        p_num.add_updater(lambda mob: mob.set_value(p.get_value() % M))

        # bottom panel: the output
        bot = Axes(x_range=[0, N, 4], y_range=[-1.3, 1.3, 1],
                   x_length=10.6, y_length=2.4,
                   axis_config={"color": anim.IRON, "include_ticks": False,
                                "stroke_width": 1.5},
                   tips=False).move_to([-0.4, -2.0, 0])
        ref = bot.plot(lambda n: np.sin(2 * np.pi * DM * n / M),
                       x_range=[0, N, 0.02], color=anim.STEEL, stroke_width=1.8)
        out_label = MathTex(r"x[n]", color=anim.RED).scale(0.6)
        out_label.move_to(bot.c2p(1.2, 1.05))
        n_label = MathTex("n").scale(0.5).next_to(bot.c2p(N, 0), RIGHT, buff=0.15)

        mode = Text("nearest neighbor", font_size=27)
        mode.move_to([3.3, -0.08, 0])

        self.play(FadeIn(top), FadeIn(stems), FadeIn(dots), FadeIn(table_label),
                  FadeIn(zero_label), FadeIn(m_label),
                  FadeIn(bot), FadeIn(ref), FadeIn(out_label), FadeIn(n_label),
                  run_time=1.2)
        self.play(FadeIn(dm_label), FadeIn(p_read), FadeIn(wrap),
                  FadeIn(wrap_label), FadeIn(mode), run_time=0.8)
        self.add(pointer)

        def run_beat(reader, ring_maker):
            outs = VGroup()
            self.add(outs)
            rings = ring_maker()
            self.add(*rings)
            for n in range(1, N + 1):
                self.play(p.animate.set_value(n * DM), run_time=0.2,
                          rate_func=smooth)
                true = np.sin(2 * np.pi * DM * n / M)
                val = reader(n * DM)
                w = Line(bot.c2p(n, true), bot.c2p(n, val),
                         color=anim.GOLD, stroke_width=2.4)
                d = Dot(bot.c2p(n, val), radius=0.06, color=anim.RED)
                outs.add(w, d)
                self.play(FadeIn(w), FadeIn(d, scale=1.6), run_time=0.12)
            self.wait(1.0)
            for r in rings:
                self.remove(r)
            return outs

        # beat one: nearest neighbor, one red ring on the entry that is read
        def nearest_rings():
            ring = always_redraw(lambda: Circle(
                radius=0.11, color=anim.RED, stroke_width=2.6).move_to(
                top.c2p(int(p.get_value()) % M, read_nearest(p.get_value()))))
            return [ring]

        outs = run_beat(read_nearest, nearest_rings)

        # beat two: interpolation, rings on both neighbors and a gold dot
        # riding the chord between them
        new_mode = Text("linear interpolation", font_size=27)
        new_mode.move_to(mode.get_center())
        self.play(FadeOut(outs), p.animate.set_value(0),
                  Transform(mode, new_mode), run_time=1.0)

        def interp_rings():
            left = always_redraw(lambda: Circle(
                radius=0.11, color=anim.TEAL, stroke_width=2.4).move_to(
                top.c2p(int(p.get_value()) % M, table[int(p.get_value()) % M])))
            right = always_redraw(lambda: Circle(
                radius=0.11, color=anim.TEAL, stroke_width=2.4).move_to(
                top.c2p((int(p.get_value()) + 1) % M,
                        table[(int(p.get_value()) + 1) % M])))
            blend = always_redraw(lambda: Dot(
                top.c2p(p.get_value() % M, read_interp(p.get_value())),
                radius=0.065, color=anim.GOLD))
            return [left, right, blend]

        run_beat(read_interp, interp_rings)
        self.wait(1.2)


anim.show(WavetableRead)